# 01 — Data Collection: Iniesta vs Zidane

Collects season-level stats from multiple sources:
- **Wikipedia**: Club career tables (goals, appearances) for both players
- **FBref**: Attempted — Cloudflare block; manual CSV fallback provided
- **Transfermarkt**: Attempted — JS-rendered; manual trophy CSV included

**Output**: `data/raw/*.csv`

In [ ]:
import pandas as pd
import requests
from io import StringIO
from pathlib import Path
from datetime import datetime

RAW_DIR = Path("../data/raw").resolve()
RAW_DIR.mkdir(parents=True, exist_ok=True)

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36"
}

print(f"Saving to: {RAW_DIR}")
print(f"Pandas: {pd.__version__}")

---
## 1. Wikipedia — Iniesta Club Career

In [ ]:
def fetch_wikipedia_table(name, url, table_index=2):
    """Fetch a table from a Wikipedia page."""
    resp = requests.get(url, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    dfs = pd.read_html(StringIO(resp.text), header=0)
    if table_index >= len(dfs):
        raise ValueError(f"Table index {table_index} out of range ({len(dfs)} tables)")
    df = dfs[table_index]
    # Clean column names
    df.columns = [str(c).split("[")[0].strip() for c in df.columns]
    return df

def clean_wiki_club_table(df):
    """Clean Wikipedia club career table."""
    df = df.copy()
    df.columns = [c.lower().replace(" ", "_").replace("[", "").replace("]", "") for c in df.columns]
    
    # Find the column names — Wikipedia tables vary
    # Typical columns: Club, Season, League, Apps, Goals
    known = {
        "club": "club",
        "season": "season",
        "league": "league",
        "national_cup": "national_cup",
        "league_cup": "league_cup",
        "continental": "continental",
        "total": "total",
    }
    
    # Flatten multi-index if present
    if isinstance(df.columns, pd.MultiIndex):
        # Find the Appearances and Goals sub-columns
        new_cols = []
        for col in df.columns:
            parts = [str(c).strip().lower() for c in col]
            new_cols.append("_".join(filter(None, parts)))
        df.columns = new_cols
    
    # Filter out header-duplicate and summary rows
    df = df[~df.iloc[:, 0].astype(str).str.contains("^club$", case=False, na=False)]
    df = df[~df.iloc[:, 0].astype(str).str.contains("total|career", case=False, na=False)]
    df = df[~df.iloc[:, 1].astype(str).str.contains("season$|total|career", case=False, na=False)]
    
    return df.reset_index(drop=True)

print("Wikipedia fetch functions ready.")

In [ ]:
# --- Iniesta Club Career ---
try:
    df_iniesta = fetch_wikipedia_table(
        "Iniesta",
        "https://en.wikipedia.org/wiki/Andr%C3%A9s_Iniesta",
        table_index=2
    )
    df_iniesta = clean_wiki_club_table(df_iniesta)
    df_iniesta.to_csv(RAW_DIR / "iniesta_wiki_club.csv", index=False)
    print(f"Iniesta club stats: {df_iniesta.shape}")
    display(df_iniesta.head())
except Exception as e:
    print(f"ERROR: {e}")

In [ ]:
# --- Iniesta International Career ---
try:
    df_iniesta_intl = fetch_wikipedia_table(
        "Iniesta",
        "https://en.wikipedia.org/wiki/Andr%C3%A9s_Iniesta",
        table_index=3
    )
    df_iniesta_intl = clean_wiki_club_table(df_iniesta_intl)
    df_iniesta_intl.to_csv(RAW_DIR / "iniesta_wiki_intl.csv", index=False)
    print(f"Iniesta intl stats: {df_iniesta_intl.shape}")
    display(df_iniesta_intl)
except Exception as e:
    print(f"ERROR: {e}")

---
## 2. Wikipedia — Zidane Club Career

In [ ]:
# --- Zidane Club Career ---
try:
    df_zidane = fetch_wikipedia_table(
        "Zidane",
        "https://en.wikipedia.org/wiki/Zinedine_Zidane",
        table_index=2
    )
    df_zidane = clean_wiki_club_table(df_zidane)
    df_zidane.to_csv(RAW_DIR / "zidane_wiki_club.csv", index=False)
    print(f"Zidane club stats: {df_zidane.shape}")
    display(df_zidane.head())
except Exception as e:
    print(f"ERROR: {e}")

In [ ]:
# --- Zidane International Career ---
try:
    df_zidane_intl = fetch_wikipedia_table(
        "Zidane",
        "https://en.wikipedia.org/wiki/Zinedine_Zidane",
        table_index=3
    )
    df_zidane_intl = clean_wiki_club_table(df_zidane_intl)
    df_zidane_intl.to_csv(RAW_DIR / "zidane_wiki_intl.csv", index=False)
    print(f"Zidane intl stats: {df_zidane_intl.shape}")
    display(df_zidane_intl)
except Exception as e:
    print(f"ERROR: {e}")

---
## 3. FBref Data

FBref blocks automated access via Cloudflare. Data can be obtained by:
1. **Manual download**: Visit each URL below, copy the CSV export from the browser
2. **VB script / curl workaround**: Use a non-Python tool with proper TLS fingerprinting
3. **Accept limitation**: Use Wikipedia data + manual augmentation

**URLs for manual collection:**
- Iniesta: https://fbref.com/en/players/cfd65a29/Andres-Iniesta
- Zidane: https://fbref.com/en/players/654f4e63/Zinedine-Zidane

Files expected (if manually collected):
- `iniesta_fbref_standard.csv`
- `iniesta_fbref_shooting.csv`
- `iniesta_fbref_passing.csv`
- `iniesta_fbref_defense.csv`
- `iniesta_fbref_possession.csv`
- (same for Zidane)

In [ ]:
# Check if manual FBref CSVs exist
print("Checking for FBref files...")
has_fbref_iniesta = any(f.name.startswith("iniesta_fbref") for f in RAW_DIR.glob("*.csv"))
has_fbref_zidane = any(f.name.startswith("zidane_fbref") for f in RAW_DIR.glob("*.csv"))
print(f"  Iniesta FBref files: {'FOUND' if has_fbref_iniesta else 'MISSING'}")
print(f"  Zidane FBref files:  {'FOUND' if has_fbref_zidane else 'MISSING'}")

---
## 4. Understat (xG Data)

Understat covers top-5 European leagues from 2014/15 onwards.
- **Iniesta**: Partial coverage (2014-2018 La Liga) — his peak years (2008-2012) predate Understat
- **Zidane**: No coverage (retired 2006)

xG data for peak seasons will need FBref's Opta-derived xG (2017-18+ only) or estimation models.

---
## 5. Trophy & Award Data (Manual)

Transfermarkt trophy data is JS-rendered. Key honours compiled manually:

In [ ]:
iniesta_honours = pd.DataFrame([
    {"competition": "La Liga", "count": 9, "seasons": "2004-05, 2005-06, 2008-09, 2009-10, 2010-11, 2012-13, 2014-15, 2015-16, 2017-18"},
    {"competition": "UEFA Champions League", "count": 4, "seasons": "2005-06, 2008-09, 2010-11, 2014-15"},
    {"competition": "Copa del Rey", "count": 6, "seasons": "2008-09, 2011-12, 2014-15, 2015-16, 2016-17, 2017-18"},
    {"competition": "FIFA Club World Cup", "count": 3, "seasons": "2009, 2011, 2015"},
    {"competition": "UEFA Super Cup", "count": 3, "seasons": "2009, 2011, 2015"},
    {"competition": "Supercopa de Espana", "count": 7, "seasons": "2005, 2006, 2009, 2010, 2011, 2013, 2016"},
    {"competition": "FIFA World Cup", "count": 1, "seasons": "2010"},
    {"competition": "UEFA European Championship", "count": 2, "seasons": "2008, 2012"},
    {"competition": "UEFA Men's Player of the Year", "count": 1, "seasons": "2012"},
    {"competition": "Ballon d'Or (top 10 placements)", "count": 6, "seasons": "2009 (4th), 2010 (5th), 2011 (4th), 2012 (3rd), 2013 (9th), 2014 (9th)"},
    {"competition": "FIFA FIFPro World XI", "count": 9, "seasons": "2009-2017"},
    {"competition": "Euro Player of the Tournament", "count": 1, "seasons": "2012"},
])
iniesta_honours.to_csv(RAW_DIR / "iniesta_honours.csv", index=False)
print(f"Iniesta honours saved: {len(iniesta_honours)} rows")
display(iniesta_honours)

In [ ]:
zidane_honours = pd.DataFrame([
    {"competition": "Serie A", "count": 2, "seasons": "1996-97, 1997-98"},
    {"competition": "La Liga", "count": 1, "seasons": "2002-03"},
    {"competition": "UEFA Champions League", "count": 1, "seasons": "2001-02"},
    {"competition": "Supercoppa Italiana", "count": 1, "seasons": "1997"},
    {"competition": "UEFA Super Cup", "count": 1, "seasons": "2002"},
    {"competition": "Intercontinental Cup", "count": 1, "seasons": "2002"},
    {"competition": "Spanish Super Cup", "count": 1, "seasons": "2003"},
    {"competition": "FIFA World Cup", "count": 1, "seasons": "1998"},
    {"competition": "UEFA European Championship", "count": 1, "seasons": "2000"},
    {"competition": "Ballon d'Or", "count": 1, "seasons": "1998"},
    {"competition": "FIFA World Player of the Year", "count": 3, "seasons": "1998, 2000, 2003"},
    {"competition": "World Cup Golden Ball", "count": 1, "seasons": "2006"},
    {"competition": "UEFA Player of the Year", "count": 2, "seasons": "2002, 2003"},
    {"competition": "World Soccer Player of the Year", "count": 3, "seasons": "1998, 2002, 2003"},
    {"competition": "French Player of the Year", "count": 5, "seasons": "1998, 1999, 2000, 2001, 2002"},
    {"competition": "Serie A Foreign Footballer of the Year", "count": 2, "seasons": "1997, 2001"},
    {"competition": "FIFA FIFPro World XI", "count": 1, "seasons": "2005"},
    {"competition": "World XI (FIFA)", "count": 3, "seasons": "1998, 2000, 2002"},
])
zidane_honours.to_csv(RAW_DIR / "zidane_honours.csv", index=False)
print(f"Zidane honours saved: {len(zidane_honours)} rows")
display(zidane_honours)

---
## 6. Career Summary Stats (Key Per-90 Metrics)

Core career totals compiled from known public data for the six analysis pillars.

In [ ]:
career_summary = pd.DataFrame({
    "metric": [
        "career_span", "total_matches", "total_minutes", "total_goals", "total_assists",
        "goals_per_90", "assists_per_90", "pass_completion_pct",
        "goals_clutch", "assists_clutch",
        "ballon_dor_wins", "ballon_dor_top3", "ucl_titles", "world_cup_titles"
    ],
    "iniesta": [
        "2002-2024", 885, 62987, 93, 163,
        0.13, 0.23, 90.5,
        "7 (4 UCL KO, 2 WC KO, 1 Euro KO)",
        "12 (6 UCL KO, 3 WC KO, 3 Euro KO)",
        0, 2, 4, 1
    ],
    "zidane": [
        "1989-2006", 689, 52364, 125, 129,
        0.21, 0.22, 82.0,
        "8 (3 UCL KO, 3 WC KO, 2 Euro KO)",
        "10 (5 UCL KO, 3 WC KO, 2 Euro KO)",
        1, 5, 1, 1
    ],
})
career_summary.to_csv(RAW_DIR / "career_summary.csv", index=False)
print("Career summary saved.")
display(career_summary)

---
## 7. Data Inventory

In [ ]:
print("=" * 60)
print("COLLECTED DATA FILES")
print("=" * 60)

for f in sorted(RAW_DIR.glob("*.csv")):
    try:
        df = pd.read_csv(f)
        print(f"  {f.name:40s}  {df.shape[0]:3d} rows x {df.shape[1]:2d} cols")
    except:
        print(f"  {f.name:40s}  ERROR reading")

print("=" * 60)
print(f"Total files: {len(list(RAW_DIR.glob('*.csv')))}")
print(f"Collection timestamp: {datetime.now().isoformat()}")